<a href="https://colab.research.google.com/github/RDGopal/IB9AU-2026/blob/main/Agents3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Working with a local LLM


In this notebook, we explore the power and advantages of running Large Language Models (LLMs) locally. While cloud-based proprietary LLMs offer convenience, they often come with significant costs, strict rate limits, and potential privacy concerns due to data being processed on external servers. By leveraging local LLMs, we gain full control over our data, ensure privacy, and eliminate ongoing API costs and rate limitations, making them an excellent choice for research, development, and applications requiring sensitive data handling. Here, we'll be working with a local Code Agent.

#The Local "Code Agent" (Qwen 2.5-Coder-3B-Instruct)
⚠️ **Important Colab Note: Ensure your runtime is set to T4 GPU (Runtime > Change runtime type > T4 GPU). This model fits tightly in the 16GB VRAM, so do not run other large models in the same session.**

Before we can run our local Code Agent, we need to install the necessary Python libraries. `smolagents` is our framework for building agents, while `transformers`, `accelerate`, and `bitsandbytes` are crucial for efficiently loading and running large language models on a GPU. `yfinance`, `seaborn`, and `matplotlib` are data science libraries the agent will use for financial data analysis and visualization.

In [ ]:
# ==========================================
# Install Dependencies
# ==========================================
# We need 'accelerate' and 'bitsandbytes' to load the model efficiently on the GPU.
# Added 'duckduckgo_search' to enable web search capabilities for the agent.
!pip install -q smolagents transformers accelerate bitsandbytes yfinance seaborn matplotlib duckduckgo_search


Now we'll load the `Qwen 2.5 Coder 3B` model. This model is specifically designed for code generation tasks. We're using `TransformersModel` from `smolagents` to load it. `device_map='auto'` ensures the model automatically utilizes the available T4 GPU, and `torch_dtype=torch.float16` reduces memory footprint by using half-precision floating-point numbers. `max_new_tokens` is set to allow for longer code outputs.

In [ ]:
# ==========================================
# Load the 3B Model (Lightweight & Fast)
# ==========================================
from smolagents import CodeAgent, TransformersModel
import torch

print("⬇️ Downloading Qwen 2.5 Coder 3B (approx 6GB)...")

# We use the 3B-Instruct model.
# device_map="auto" finds the Colab GPU.
# torch_dtype=torch.float16 cuts memory usage in half.
model = TransformersModel(
    model_id="Qwen/Qwen2.5-Coder-3B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16,
    max_new_tokens=2048  # Give it space to write longer scripts
)

print("✅ 3B Model loaded on GPU! Ready for coding.")

With the model loaded, we can now initialize our `CodeAgent`. We provide it with the loaded `model` and specify `additional_authorized_imports`. This list tells the agent which Python libraries it is allowed to use when generating and executing code. Crucially, we don't give it predefined tools; instead, it writes its own code using these authorized libraries, making it highly flexible.

In [ ]:
# ==========================================
# Initialize the Agent
# ==========================================

# We authorize the agent to use specific data science libraries, and now also a web search library.
agent = CodeAgent(
    tools=[], # We don't need pre-made tools; the agent writes its own code.
    model=model,
    max_steps=3,
    additional_authorized_imports=[
        "yfinance",
        "pandas",
        "numpy",
        "seaborn",
        "matplotlib.pyplot",
        "duckduckgo_search" # Added for web search capabilities
    ]
)


#Coding task
*Download data for BTC-USD, ETH-USD, and SOL-USD for the last 90 days. Calculate the correlation matrix and plot it as a heatmap.*

Here, we're giving the `CodeAgent` a multi-step task to perform. It involves downloading financial data for cryptocurrencies using `yfinance`, calculating correlations, and then visualizing the results as a heatmap using `seaborn`. The agent will interpret this prompt, write the Python code, and then execute it to produce the `crypto_heatmap.png` file.

In [ ]:
# ==========================================
# Execute the Crypto Heatmap Task
# ==========================================
# We give it a complex multi-step instruction.
task_prompt = """
1. Download daily closing data for 'BTC-USD', 'ETH-USD', and 'SOL-USD' for the last 30 days using yfinance.
2. Create a single DataFrame with these close prices.
3. Calculate the correlation matrix of the returns.
4. Plot this correlation matrix as a heatmap using seaborn with annotations.
5. Save the plot as 'crypto_heatmap.png'.
"""

print("🤖 Agent is coding... (Watch the 'Thought' process below)")
result = agent.run(task_prompt,stream=False)

In [ ]:
print(result)

After the agent has attempted to generate and save the heatmap, this code block checks if the `crypto_heatmap.png` file exists. If it does, it uses `IPython.display` to show the generated image directly within the Colab output. This allows us to visually inspect the agent's output.

In [ ]:
# ==========================================
# Display the Result
# ==========================================
import IPython
import os

if os.path.exists("crypto_heatmap.png"):
    print("\n📊 Displaying Generated Heatmap:")
    IPython.display.display(IPython.display.Image("crypto_heatmap.png"))
else:
    print("⚠️ No image file found. Check the agent's output logs above for errors.")

#Let's replicate previous tasks with the local LLM


This task demonstrates the agent's ability to perform data analysis using `pandas`. We ask it to fetch historical stock data for NVIDIA (`NVDA`), calculate the daily percentage returns, and then determine the standard deviation (volatility) of these returns. The agent will write the necessary `yfinance` and `pandas` code to achieve this.

In [ ]:
# The "Pandas" Task
# We ask for a calculation that requires fetching data and doing math.

task = """
Get the historical closing prices for 'NVDA' for the last 1 month.
Calculate the standard deviation of the daily percentage returns (volatility).
Print the volatility as a percentage.
"""

result=agent.run(task,stream=False)

In [ ]:
print(result)

In this visual task, the agent is instructed to fetch Bitcoin (BTC-USD) closing prices, calculate a 7-day moving average, and then plot both on a chart using `matplotlib`. The goal is to see if the agent can generate a complete visualization and save it as an image file (`btc_chart.png`). A helper snippet is included to display the image if the agent doesn't do so automatically.

In [ ]:
# The "Visual" Task
# The agent will use matplotlib to create a chart.

task_viz = """
Plot the closing price of Bitcoin (BTC-USD) for the last 3 months.
Add a 7-day moving average line to the chart.
Save the chart as 'btc_chart.png' and display it.
"""

agent.run(task_viz)

# Helper to show the image in Colab if the agent doesn't auto-display
import IPython
if os.path.exists("btc_chart.png"):
    IPython.display.display(IPython.display.Image("btc_chart.png"))

This task challenges the agent to act as a technical analyst. It needs to calculate the Relative Strength Index (RSI), a common momentum indicator, for Apple stock (`AAPL`) over a 14-day period. Based on the calculated RSI value, it should then determine if the stock is 'Overbought' or 'Oversold' according to standard thresholds.

In [ ]:
# The Technical Analyst Task
task = """
Prompt: Calculate the Relative Strength Index (RSI) for Apple stock over the
last 14 days and tell me if it is currently 'Overbought' (>70) or 'Oversold' (<30).
"""

result = agent.run(task,stream=False)

In [ ]:
print(result)

This final task tests the agent's ability to combine information retrieval and data analysis. It's asked to first find the date of the next Federal Reserve meeting (which would typically involve a search tool, but here it's expected to generate code that could query an API or web scrape if it were set up for that, though in this context it's more about writing the *logic*). Then, it needs to calculate the volatility of the S&P 500 (`SPY`) around previous Fed meeting dates.

In [ ]:
# Macro Researcher Task
task = """
Prompt: Search for the date of the next Federal Reserve meeting.
Then, calculate the average volatility of the S&P 500 (SPY) during the week of the previous 3 Fed meetings.
"""

result = agent.run(task,stream=False)

In [ ]:
print(result)

#Required Task 17

**The Assignment:** You are a Quantitative Analyst. Your boss wants to know which sector has performed better this year on a risk-adjusted basis: Big Tech or Big Banks.

**Task:** Write a prompt for your Local smolagents (Qwen 3B) agent to perform the following steps autonomously:

**Data Ingestion:** Download daily closing prices for the last 180 days for a Tech Portfolio (NVDA, AAPL, MSFT) and a Bank Portfolio (JPM, BAC, C).

**Financial Math:**

*   Calculate the Daily Returns for each stock.
*   Calculate the Sharpe Ratio for each stock (Assume risk-free rate = 0, so simply Mean Daily Return / Std Dev of Daily Returns * sqrt(252)).

**Visualization:**

* Create a Bar Chart comparing the Sharpe Ratios of all 6 companies.

* Color code the bars: Green for Tech, Blue for Banks.

**Output:** Save the chart as `sharpe_comparison.png`.
